# Read dictionary values in the ERD payload.

Run the setup first, then each step in order. This notebook executes the same declarations, calls and assertions as the [script](09_erd.py). Outputs were captured by direct Python execution in a fresh process. No Jupyter kernel was used.


In [1]:
"""Read dictionary values in the ERD payload.
Run: .venv/bin/python examples/step_20_dictionaries/09_erd.py


Follow one dictionary through the actual Maxitor query.
"""

from typing import Annotated

from pydantic import Field

from aoa.action_machine.domain import AssociationOne, BaseCoreEntity, BaseEnumEntity, NoInverse, value
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


class ShopDomain(BaseDomain):
    """Group the shop declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Currencies accepted by the shop", domain=ShopDomain)
class CurrencyEnumEntity(BaseEnumEntity):
    """Declare the shop's currency vocabulary."""

    symbol: str = Field(description="Currency symbol")

    EUR = value(code="eur", display_name="Euro", symbol="€")
    USD = value(code="usd", display_name="US dollar", symbol="$")


@entity(description="Price tag", domain=ShopDomain)
class PriceTagCoreEntity(BaseCoreEntity):
    """Associate an amount with its currency."""

    id: str = Field(description="Tag identifier")
    amount: float = Field(description="Price amount")
    currency: Annotated[AssociationOne[CurrencyEnumEntity], NoInverse()] = Field(description="Money unit")

## Load the integration API


In [2]:
import json

## Load the integration API


In [3]:
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## Load the integration API


In [4]:
from aoa.maxitor.model.diagrams.actions.list_entities_action import ListEntitiesAction

## Load the integration API


In [5]:
from aoa.maxitor.model.diagrams.resources.duckdb_graph_resource import DuckDBGraphResource

## Run the next step


In [6]:
machine = ActionProductMachine(loggers=[])

## Run the next step


In [7]:
domain = next(node.node_id for node in machine.graph_coordinator.get_all_nodes() if node.label == "ShopDomain")

## Run the next step


In [8]:
store = DuckDBGraphResource.build_from_json(json.loads(machine.graph_coordinator.to_json()))

## Load the integration API


In [9]:
import asyncio

## Load the integration API


In [10]:
from aoa.action_machine.model import BaseState

## Load the integration API


In [11]:
from aoa.maxitor.model.diagrams.resources.duckdb_graph_resource import DUCKDB_GRAPH_CONNECTION_KEY

## Run the next step


In [12]:
result = asyncio.run(
    ListEntitiesAction().build_domain_payload_summary(
        ListEntitiesAction.Params(domain_qualnames=[domain], include_one_hop_neighbors=False),
        BaseState(),
        None,
        {DUCKDB_GRAPH_CONNECTION_KEY: store},
    )
)

## Run the next step


In [13]:
diagram = result.model_dump(mode="json")["domain_slices"][0]["list_entities"]

## Run the next step


In [14]:
currency = next(item for item in diagram["entities"] if item["label"] == "CurrencyEnumEntity")

## Run the next step


In [15]:
tag = next(item for item in diagram["entities"] if item["label"] == "PriceTagCoreEntity")

## Run the next step


In [16]:
relation = next(item for item in diagram["relations"] if item["relationship_kind"] == "association")

## Verify the result


In [17]:
assert relation["relationship_kind"] == "association"

## Inspect the result


In [18]:
print(currency["values"][0])

{'code': 'eur', 'display_name': 'Euro', 'properties': [{'name': 'symbol', 'value': '€'}]}


## Inspect the result


In [19]:
print(next(field["reference_qualname"] for field in tag["fields"] if field["name"] == "currency").split(".")[-1])

CurrencyEnumEntity


## Inspect the result


In [20]:
print("Currency field rows:", sum(field["name"] == "currency" for field in tag["fields"]))

Currency field rows: 2


## Inspect the result


In [21]:
print("Relation:", relation["label"], "->", relation["target"].split(".")[-1])

Relation: currency (one) -> CurrencyEnumEntity
